# Exercice pratique : extraction d'informations avec un LLM 🤖

## Des récits de voyage aux données géolocalisées 📍

### Objectif 🎯

Utiliser un grand modèle de langage (LLM), via une API cloud ou une installation locale, pour extraire des lieux de récits de voyage non structurés, puis enrichir les données extraites avec leurs coordonnées à l'aide du service de géocodage de la Base Adresse Nationale (BAN).

### Données d'entrée 📂

Le dossier `data/texts/`, situé par rapport à ce notebook, contient six récits de voyage en français. Chacun décrit une semaine dans un département différent de la région Provence-Alpes-Côte d'Azur. Les récits mentionnent différents types de lieux : villes, monuments, quartiers et sites naturels.

### Partie 1 — Extraire les lieux avec un LLM 📍

1. Lisez les six fichiers `.txt` et traitez chaque récit avec l'API LLM de votre choix. Gardez votre clé API en dehors du notebook et ne l'affichez ni ne la partagez jamais.
2. Demandez au modèle d'identifier les lieux mentionnés dans chaque récit. Pour chaque lieu, extrayez au minimum son nom, son type, son département et, lorsque le texte le précise ou l'associe clairement au lieu, sa commune.
3. Distinguez les lieux visités par le voyageur de ceux qui ont seulement été vus, mentionnés ou envisagés. Consignez cette information lorsque le texte le permet ; sinon, utilisez une valeur indiquant qu'elle est inconnue. Ne comptez pas plusieurs fois les mentions répétées d'un même lieu dans un récit.
4. Demandez une sortie JSON structurée. Définissez et utilisez un schéma pour valider la réponse. Incluez le nom du fichier source afin de pouvoir retrouver le récit d'origine. Utilisez `null` pour les informations impossibles à établir à partir du texte ; ne demandez pas au modèle d'inventer des coordonnées.
5. Enregistrez les résultats extraits au format JSON. Vérifiez un échantillon à partir des récits d'origine et notez les erreurs d'extraction.

#### Exemple de structure JSON extraite 🧩

Utilisez cette structure comme modèle. L'exemple ne présente qu'un lieu d'un seul récit ; votre résultat complet doit inclure les lieux extraits des six récits. `visit_status` peut prendre les valeurs `visited`, `seen`, `mentioned`, `considered` ou `unknown`.


Exemple de résultat extrait :

```json
[
  {
    "source_file": "voyage_alpes_de_haute_provence.txt",
    "places": [
      {
        "name": "Digne-les-Bains",
        "type": "municipality",
        "department": "Alpes-de-Haute-Provence",
        "municipality": "Digne-les-Bains",
        "visit_status": "visited"
      }
    ]
  }
]
```


### Partie 2 — Enrichir le JSON avec le géocodage 📍

1. Lisez le JSON produit dans la partie 1. Pour chaque lieu extrait, construisez une requête de recherche à partir de son nom et de tout contexte géographique disponible, comme sa commune et son département.
2. Interrogez l'[API de géocodage Géoplateforme](https://cartes.gouv.fr/aide/fr/guides-utilisateur/utiliser-les-services-de-la-geoplateforme/geocodage/), qui donne accès aux données d'adresses de la BAN. Consultez la documentation pour choisir les paramètres de recherche adaptés. Le service propose aussi un index de points d'intérêt ; demandez-vous s'il convient aux types de lieux recherchés.
3. Examinez les résultats proposés avant de choisir une correspondance. Ajoutez à chaque lieu sa longitude, sa latitude, le libellé correspondant et, si elle existe, la précision du résultat. Indiquez si la correspondance est acceptée, incertaine ou introuvable.
4. Si aucun résultat ne convient, laissez les coordonnées à `null` ; ne les remplacez pas silencieusement par le centre d'une commune lorsqu'il s'agit d'un site précis. Gérez explicitement les erreurs de l'API et évitez les requêtes répétées inutiles.
5. Enregistrez les résultats enrichis dans un nouveau fichier JSON, en conservant les informations extraites d'origine et le nom du fichier source.

#### Exemple de lieu enrichi 🗺️

Ajoutez un objet `geocoding` à chaque lieu. L'exemple ci-dessous illustre un lieu pour lequel aucune correspondance pertinente n'a été trouvée ; il ne prétend pas décrire les résultats réels du géocodeur.


Exemple d'objet de lieu enrichi :

```json
{
  "name": "Digne-les-Bains",
  "type": "municipality",
  "department": "Alpes-de-Haute-Provence",
  "municipality": "Digne-les-Bains",
  "visit_status": "visited",
  "geocoding": {
    "status": "not_found",
    "matched_label": null,
    "score": null,
    "longitude": null,
    "latitude": null
  }
}
```


Utilisez `accepted`, `uncertain` ou `not_found` pour indiquer le statut du géocodage. Pour une correspondance acceptée, renseignez le libellé disponible, la précision et les coordonnées numériques. Pour une correspondance incertaine, conservez séparément les détails du candidat ; les coordonnées confirmées doivent rester à `null` tant que la correspondance n'a pas été vérifiée.


### Discussion 💭

- Quels types de lieux étaient faciles ou difficiles à extraire ? Le modèle a-t-il confondu des lieux simplement mentionnés avec des lieux réellement visités ?
- Certains noms étaient-ils ambigus, absents du géocodeur ou associés au mauvais lieu ? En quoi le contexte géographique et les scores de correspondance ont-ils aidé ?
- Pouvez-vous être certains que les coordonnées renvoyées correspondent au site précis plutôt qu'à une adresse ou à une zone plus large ? Comparez plusieurs résultats avec une carte fiable.


### Livrables 📦

- Votre travail d'extraction et de géocodage dans ce notebook.
- Le JSON extrait et le JSON enrichi.
- Une brève discussion des résultats, des erreurs et des limites. N'incluez pas votre clé API dans une remise.


### Étapes suivantes 🚀

- Écrivez votre code de façon à pouvoir l'exécuter facilement depuis la ligne de commande, avec les chemins des fichiers d'entrée et de sortie comme paramètres.
- Conteneurisez l'application avec Docker pour la rendre portable et reproductible.
